# AGENTICAITA: Deliberative Multi-Agent Reasoning for Autonomous Trading
**ArXivist-generated reproduction notebook**  
Paper: [arxiv:2605.12532](https://arxiv.org/abs/2605.12532)  
Generated: 2026-05-16

This notebook walks through the key components of AGENTICAITA, demonstrates each module 
with synthetic data, runs a dry-run pipeline simulation, and shows how to reproduce the 
paper's reported metrics.

In [ ]:
import sys
print(f'Python: {sys.version}')

# Check key dependencies
try:
    import numpy as np
    print(f'NumPy: {np.__version__}')
except ImportError:
    print('NumPy not found — run: pip install -r ../requirements.txt')

try:
    import pydantic
    print(f'Pydantic: {pydantic.__version__}')
except ImportError:
    print('Pydantic not found — run: pip install -r ../requirements.txt')

print('\nNo GPU required — AGENTICAITA is a training-free system.')
print('LLM inference is delegated to Ollama (external process).')

In [ ]:
import subprocess
result = subprocess.run(['pip', 'install', '-q', '-e', '..'], capture_output=True, text=True)
if result.returncode == 0:
    print('Package installed successfully.')
else:
    print('Installation output:', result.stderr[:500])

## Paper Overview

AGENTICAITA replaces the traditional signal-then-execute trading paradigm with a **deliberative 
multi-agent loop** in which three specialized LLM agents reason, negotiate, and act — with no 
offline training or human intervention.

**Key innovation**: Rather than asking a single LLM to simultaneously analyze the market, assess 
risk, and decide on execution (which induces confirmation bias), decisions are decomposed into 
a chain of three agents with *distinct epistemic mandates*:

1. **Analyst** — market analysis → trading signal (JSON)
2. **Risk Manager** — 4 hard gates + LLM validation (JSON)
3. **Executor** — DRY_RUN logging or LIVE order routing

The system is activated selectively by the **AZTE** (Z-score trigger) and serialized by the 
**IGP** (mutex scheduler), with portfolio diversity enforced via the **CBD** composite score.

**Validated**: 5-day dry-run, 157 invocations, 76 assets, zero human interventions,  
11.5% agentic friction rate, +14.94pp alpha vs BTC buy-and-hold.

## Module 1: AZTE — Adaptive Z-Score Trigger Engine

**Paper Section 4.1, Equations 1–3**

The AZTE is the *cognitive resource allocator*. It gates LLM inference exclusively on 
statistically anomalous market conditions via a rolling Z-score baseline.

$$r_t = \left|\frac{p_t - p_{t-1}}{p_{t-1}}\right| \quad \text{(Eq. 1)}$$

$$z_t = \frac{r_t - \hat{\mu}_W}{\hat{s}_W} \quad \text{(Eq. 2)}$$

$$\mathcal{T}_t = \mathbf{1}[z_t \geq 2.0] \vee \mathbf{1}[r_t \geq 0.003] \quad \text{(Eq. 3)}$$

**Key property — Regime Invariance**: the 2σ threshold self-adapts to prevailing volatility 
without manual recalibration. W=30 bars = 30 minutes at 60s polling frequency.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'src'))

try:
    from agenticaita.trigger.azte import AZTE
    from agenticaita.utils.config import TriggerConfig
    import asyncio
    import numpy as np

    config = TriggerConfig(z_threshold=2.0, r_floor=0.003, window_bars=30, per_asset_cooldown_s=300)
    azte = AZTE(config)

    # Simulate 35 ticks of price data for BTC
    np.random.seed(42)
    prices = [84000.0]
    for _ in range(34):
        prices.append(prices[-1] * (1 + np.random.normal(0, 0.001)))

    # Inject a spike at tick 35
    prices.append(prices[-1] * 1.008)  # +0.8% spike → should trigger

    async def demo_azte():
        triggers = []
        for i, p in enumerate(prices):
            evt = await azte.update('BTC', p)
            if evt:
                triggers.append((i, evt))
        return triggers

    triggers = asyncio.run(demo_azte())

    print(f'Fed {len(prices)} price ticks to AZTE')
    print(f'Triggers fired: {len(triggers)}')
    for tick_idx, evt in triggers:
        print(f'  Tick {tick_idx}: z={evt.z_score:.3f}, r={evt.return_magnitude:.4f}, by={evt.triggered_by!r}')
    print(f'\nAZTE: {azte}')

except Exception as e:
    print(f'Error: {e} — ensure the package is installed with pip install -e ..')

## Module 2: CBD — Correlation-Break Diversification

**Paper Section 4.5, Equations 9–11, Proposition 1**

The CBD composite score rewards assets that are both statistically anomalous *and* 
decorrelated from BTC, operationalizing portfolio diversity within the Analyst's reasoning.

$$\rho^a_{cb} = 1 - |\rho(\text{prices}_a, \text{prices}_{BTC})| \quad \text{(Eq. 9)}$$

$$\tilde{z}^a_t = \left(1 - e^{-\kappa(|z^a_t| - 2.0)}\right) \cdot \mathbf{1}[|z^a_t| \geq 2.0] \quad \text{(Eq. 10)}$$

$$\Omega^a = \underbrace{\alpha \tilde{z}^a_t}_{\text{anomaly}} + \underbrace{(1-\alpha) \rho^a_{cb}}_{\text{decorrelation}}, \quad \alpha=0.5 \quad \text{(Eq. 11)}$$

**Proposition 1**: For equal anomaly magnitude, the more BTC-decorrelated asset gets higher Ω. 
An asset perfectly correlated with BTC (|ρ|=1) gets no diversification bonus.

In [ ]:
try:
    from agenticaita.scoring.cbd import CBD
    from agenticaita.utils.config import CBDConfig
    import numpy as np

    cbd = CBD(CBDConfig(alpha=0.5, kappa=0.5, window_bars=30))

    np.random.seed(0)
    btc_prices = list(84000.0 * np.cumprod(1 + np.random.normal(0, 0.001, 30)))

    # High-corr asset: tracks BTC closely
    corr_prices = [p * (1 + np.random.normal(0, 0.0002)) for p in btc_prices]
    # Low-corr asset: independent random walk
    decorr_prices = list(0.85 * np.cumprod(1 + np.random.normal(0, 0.002, 30)))

    z_t = 2.8  # Same Z-score for both

    omega_corr   = cbd.score(z_t, corr_prices, btc_prices)
    omega_decorr = cbd.score(z_t, decorr_prices, btc_prices)

    rho_cb_corr   = cbd.decorrelation_score(corr_prices, btc_prices)
    rho_cb_decorr = cbd.decorrelation_score(decorr_prices, btc_prices)
    z_tilde       = cbd._normalized_anomaly_internal(z_t)

    print('CBD Demonstration (Proposition 1 verification)')
    print(f'Same z-score: {z_t}, z̃ = {z_tilde:.4f}')
    print()
    print(f'BTC-correlated asset:   ρ_cb={rho_cb_corr:.4f}  →  Ω={omega_corr:.4f}')
    print(f'BTC-decorrelated asset: ρ_cb={rho_cb_decorr:.4f}  →  Ω={omega_decorr:.4f}')
    print()
    assert omega_decorr > omega_corr, 'Proposition 1 violated!'
    print('✓ Proposition 1 holds: decorrelated asset has higher Ω')

except Exception as e:
    print(f'Error: {e}')

## Module 3: Risk Manager Hard Gates

**Paper Section 4.2, Equations 4–7**

The Risk Manager's Layer A applies four **deterministic hard gates** before any LLM call. 
This is an *architectural guarantee* — no LLM reasoning can override gate failure.

| Gate | Condition | Equation |
|------|-----------|----------|
| Signal validity | signal ∈ {long, short} | Eq. 4 |
| Minimum confidence | confidence ≥ 0.60 | Eq. 5 |
| Max risk per trade | \|entry - SL\| / entry ≤ 0.02 | Eq. 6 |
| Max position size | size_usd ≤ $500 | Eq. 7 |

In [ ]:
try:
    from agenticaita.agents.risk_manager import RiskManagerAgent
    from agenticaita.pipeline.contracts import AnalystContract
    from agenticaita.utils.config import LLMConfig, RiskManagerConfig

    rm = RiskManagerAgent(LLMConfig(), RiskManagerConfig())

    test_cases = [
        # (description, contract_kwargs, expected_pass)
        ('Good signal',         dict(signal='long',  confidence=0.75, entry_price=100.0, stop_loss=98.5, take_profit=103.0, size_usd=300.0, reasoning='strong signal'), True),
        ('Too low confidence',  dict(signal='long',  confidence=0.45, entry_price=100.0, stop_loss=98.5, take_profit=103.0, size_usd=300.0, reasoning='weak'), False),
        ('Risk too high',       dict(signal='short', confidence=0.80, entry_price=100.0, stop_loss=105.0, take_profit=90.0, size_usd=300.0, reasoning='risky'), False),
        ('Position too large',  dict(signal='long',  confidence=0.80, entry_price=100.0, stop_loss=98.5, take_profit=103.0, size_usd=750.0, reasoning='large'), False),
    ]

    print('Risk Manager Hard Gate Tests')
    print(f'{"Description":<25} {"Expected":<10} {"Got":<10} {"Gate"}')
    print('-' * 65)
    for desc, kwargs, expected in test_cases:
        signal = AnalystContract(**kwargs)
        result = rm.hard_gates(signal)
        status = '✓' if result.passed == expected else '✗ FAIL'
        print(f'{desc:<25} {str(expected):<10} {str(result.passed):<10} {result.failed_gate or "—"}  {status}')

except Exception as e:
    print(f'Error: {e}')

## Module 4: Agentic Friction (Eq. 8)

**Paper Section 4.3, Definition 1**

Agentic Friction quantifies inter-agent negotiation — evidence that the multi-agent 
architecture produces genuinely different outcomes from a single-agent pass-through:

$$F = \frac{N_{rej} + N_{wait}}{N} \quad \text{(Eq. 8)}$$

Paper result: N=157, N_wait=13 (8.3%), N_rej=5 (3.2%), **F = 11.5%**

In [ ]:
from agenticaita.evaluation.metrics import compute_metrics
import math

# Reproduce paper Table 5 values
N, N_wait, N_rej = 157, 13, 5
F = (N_rej + N_wait) / N
print(f'Paper values: N={N}, N_wait={N_wait}, N_rej={N_rej}')
print(f'Agentic Friction F = ({N_rej} + {N_wait}) / {N} = {F:.4f} ({F*100:.1f}%)')
print(f'Paper reports: 11.5% — match: {abs(F - 0.115) < 0.001}')

# Simulate a small session with synthetic PnL data
import numpy as np
np.random.seed(42)
n_trades = 20
pnl_series = list(np.random.normal(0.05, 1.2, n_trades))
signals = ['long'] * 18 + ['short'] * 2

metrics = compute_metrics(pnl_series, signals, n_total=25, n_wait=3, n_rejected=2)
print(f'\nSynthetic session metrics:')
print(metrics)

## Transaction Cost Sensitivity (Table 7)

**Paper Section 6, Limitation L1**

The DRY_RUN suppresses order placement, so real slippage and fees are not modelled inline. 
The paper applies costs retroactively using the round-trip model (Eq. 13).

Paper baseline: net PnL = –$15.07 on 139 trades, mean position = $188

In [ ]:
from agenticaita.evaluation.cost_model import sensitivity_analysis, benchmark_comparison

# Reproduce Table 7
scenarios = sensitivity_analysis(
    net_pnl_dryrun=-15.07,
    n_trades=139,
    mean_position_usd=188.0,
)
print('Table 7 reproduction:')
print(f'{"Scenario":<15} {"Round-trip":<12} {"Total cost":<14} {"Adj net PnL"}')
print('-' * 55)
for name, data in scenarios.items():
    print(f'{name:<15} {data["roundtrip_label"]:<12} ${abs(data["total_cost_usd"]):<13.2f} ${data["adj_net_pnl_usd"]:.2f}')

print()
# Table 4
bench = benchmark_comparison(system_pnl=-15.07, btc_pnl=-3912.0)
print('Table 4 reproduction:')
print(f'AGENTICAITA net PnL:   ${bench["agenticaita"]["net_pnl"]:.2f}')
print(f'BTC buy-and-hold:      ${bench["btc_buyhold"]["net_pnl"]:.2f}')
print(f'Alpha vs BTC:          +{bench["alpha_vs_btc_pp"]:.2f}pp')

## Full Pipeline Dry-Run Demo (Synthetic Data)

This cell runs a mini dry-run of the full pipeline using the mock market feed 
(no Ollama LLM required — the Analyst is mocked for offline testing).

In [ ]:
import asyncio
from agenticaita.data.mock_feed import MockMarketFeed
from agenticaita.trigger.azte import AZTE
from agenticaita.pipeline.igp import IGP
from agenticaita.utils.config import TriggerConfig, IGPConfig

ASSETS = ['BTC', 'ETH', 'FARTCOIN', 'XPL']

async def mini_dryrun(n_ticks=50):
    feed = MockMarketFeed(assets=ASSETS, seed=42)
    azte = AZTE(TriggerConfig(per_asset_cooldown_s=0))  # no cooldown for demo
    igp = IGP(IGPConfig(global_cooldown_s=0))
    
    n_triggers = 0
    n_busy = 0
    
    for tick in range(n_ticks):
        spike = ASSETS[tick % len(ASSETS)] if tick % 10 == 9 else None
        feed.tick(spike_asset=spike)
        
        for asset in ASSETS:
            price = feed.get_price(asset)
            event = await azte.update(asset, price)
            if event:
                n_triggers += 1
                admitted = await igp.try_acquire(asset)
                if admitted:
                    omega = 0.0  # CBD would run here in full system
                    print(f'  Tick {tick:3d} | {asset:<10} | z={event.z_score:.2f} | by={event.triggered_by}')
                    igp.release(asset)
                else:
                    n_busy += 1

    print(f'\nSummary: {n_ticks} ticks, {n_triggers} triggers, {n_busy} pipeline_busy discards')
    print(f'IGP stats: {igp.stats}')

asyncio.run(mini_dryrun())

## Paper Results

Reference results from the 5-day live dry-run (April 6–11, 2026):

In [ ]:
paper_results = {
    'total_invocations': 157,
    'unique_assets': 76,
    'human_interventions': 0,
    'trades_executed': 139,
    'win_rate': '51.80%',
    'net_pnl_usd': -15.07,
    'net_pnl_pct': '-0.058% on $26,079 notional',
    'profit_factor': 0.841,
    'max_drawdown_usd': 32.30,
    'agentic_friction_F': '11.5%',
    'long_signal_rate': '90.4%',
    'alpha_vs_btc_buyhold': '+14.94pp',
    'binomial_pvalue_H0_wr050': 0.34,
    'statistically_significant_edge': False,
    'note': 'n=139 insufficient for edge claim; 500+ trades over 90 days required'
}

print('Paper reported results (Table 3 + Table 5):')
for k, v in paper_results.items():
    print(f'  {k:<35} {v}')

print('\nTo reproduce: run scripts/run_dryrun.py, then scripts/evaluate.py')

## What To Do Next

1. **Run a dry-run session** (requires Ollama with qwen3:8b):
   ```bash
   ollama pull qwen3:8b
   python scripts/run_dryrun.py --ticks 500 --log-level INFO
   ```

2. **Evaluate session metrics**:
   ```bash
   python scripts/evaluate.py --db data/agenticaita.db --cost-scenario realistic
   ```

3. **Audit the pipeline log**:
   ```bash
   python scripts/replay.py --db data/agenticaita.db
   ```

4. **LIVE mode** (implement your DEX adapter first):
   - Subclass `MarketFeed` in `src/agenticaita/data/market_feed.py`
   - Implement `execution/routing.py` for your DEX
   - Set `execution.mode: LIVE` in `configs/default.yaml`

---

**Key implementation notes (from SIR):**

- DEX exchange not identified in paper (conf: 0.45) — `MockMarketFeed` used by default
- `Agno` orchestration framework not open-sourced (conf: 0.55) — reimplemented as asyncio loop
- LLM long-directional bias (90.4% long signals) is a documented limitation, not a code bug
- All 10 paper equations implemented exactly — see inline `# Eq. N` comments in source
- Transaction costs NOT modelled in DRY_RUN; apply `cost_model.sensitivity_analysis()` retroactively